# 02 · Data cleaning
Only defensible transformations. Every step is logged with the number of rows affected; raw files are untouched and
cleaned tables are written to `data/processed/`.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
import numpy as np, pandas as pd
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4g}")
from src import pipeline as P
from src.config import *

In [2]:
ctx = P.prepare('stage_clean')
ctx = P.stage_clean(ctx)
ctx['cleaning_log_ifood']

[19:40:41] clean


,step,rows_affected,detail
0,start,2240,raw rows
1,parse Dt_Customer,2240,ISO date -> datetime
2,Marital_Status normalisation,7,Alone->Single; Absurd/YOLO->Unknown
3,drop exact duplicates (all fields but ID),183,"identical on 28 fields incl. spend, dates, outcomes -> duplicate registrations"
4,exclude conflicting-label duplicate groups,38,same customer profile recorded with both Response=0 and 1; true label unknowable
5,"Income 666,666 placeholder -> NaN",1,flagged
6,Income missing (incl. invalid),25,kept as NaN in clean table; imputed inside modelling pipelines only
7,Year_Birth < 1920 -> age NaN,3,flagged
8,drop constant Z_ columns,0,"Z_CostContact=3, Z_Revenue=11 moved to config economics"
9,final,2019,analytical customer rows


In [3]:
ctx['cleaning_log_hill']

,step,rows_affected,detail
0,start,64000,raw rows
1,zip_code typo,28776,Surburban -> Suburban
2,derive,64000,"treatment label, history_band, recency_band, buyer_type, customer_idx (row id)"
3,final,64000,no rows removed


### Decisions
| Issue | Treatment | Why |
|---|---|---|
| 183 exact duplicates (all fields but ID, after label normalisation) | keep lowest ID | duplicate registrations inflate counts and response |
| 19 conflicting-label groups (38 rows) | exclude | the true outcome is unknowable; keeping either copy would invent a label |
| Income = 666,666 | set to NaN + flag | placeholder value; would dominate any income statistic |
| 24 missing incomes | NaN in clean table; median-imputed **inside** model pipelines (fit on training data only) | no information leakage from test data |
| Year_Birth 1893/1899/1900 | age NaN + flag | impossible ages |
| Marital 'Alone' / 'Absurd' / 'YOLO' | Single / Unknown / Unknown — done **first**, so label variants cannot hide duplicates | 7 rows, non-standard labels |
| Z_CostContact, Z_Revenue | removed as features, kept as economics parameters | constant columns |
| Hillstrom 'Surburban' | 'Suburban' | typo |

Robustness notebook 11 re-runs key results on the raw 2,240 rows to show the de-duplication does not drive conclusions.

In [4]:
clean = ctx['clean_ifood']
print('clean customers:', len(clean), '| response rate raw vs clean:', round(ctx['raw_ifood'].Response.mean(),4), round(clean.Response.mean(),4))
clean[['income_invalid_flag','income_missing_flag','birth_year_invalid_flag']].sum()

clean customers: 2019 | response rate raw vs clean: 0.1491 0.1456


,0
income_invalid_flag,1
income_missing_flag,25
birth_year_invalid_flag,3
